In [16]:
import torch
import os
import glob
from PIL import Image
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- IMPOSTAZIONE DEL DEVICE (GPU o CPU) ---
# Questa cella definisce il device una volta per tutte.
# Tutte le altre parti del codice useranno questa variabile.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device in uso: {device}")
print(f"Current working directory: {os.getcwd()}")

COLAB = False;
if COLAB:
    # Montiamo Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # Definiamo il percorso principale
    DATASET_PATH = "/content/drive/MyDrive/Uni/Magistrale/colab/1/ML/Project/training/train"
else:
    DATASET_PATH = "../../dataset/training/train"


DATASET_MEAN = [0.4278, 0.4464, 0.4231] 
DATASET_STD = [0.2543, 0.2535, 0.2567]


Device in uso: cuda
Current working directory: d:\MLGit\autoencoder-onboard-segmentation\notebooks\PreTrainedModel


In [17]:
from torchvision import transforms
import torchvision.transforms.functional as TF

# Ridefiniamo la classe Dataset per includere le trasformazioni
class SegmentationDataset(Dataset):
    def __init__(self, sample_dirs, output_size, mean, std, is_train=True):
        self.sample_dirs = sample_dirs
        self.output_size = output_size
        # Definiamo le trasformazioni direttamente qui
        self.normalize = transforms.Normalize(mean=mean, std=std)
        self.is_train = is_train

    def __len__(self):
        return len(self.sample_dirs)

    def __getitem__(self, idx):
        sample_dir = self.sample_dirs[idx]
        img_path = os.path.join(sample_dir, 'rgb.jpg')
        mask_path = os.path.join(sample_dir, 'labels.png')
        
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        # Ridimensionamento
        image = TF.resize(image, self.output_size, interpolation=TF.InterpolationMode.BILINEAR)
        mask = TF.resize(mask, self.output_size, interpolation=TF.InterpolationMode.NEAREST)
        
        # --- DATA AUGMENTATION (solo per il training set) ---
        if self.is_train:
            # Esempio: flip orizzontale con probabilità 50%
            if torch.rand(1) < 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)

        # Conversione a tensore e normalizzazione
        image = TF.to_tensor(image)
        image = self.normalize(image)
        mask = torch.from_numpy(np.array(mask, dtype=np.int64))

        return image, mask
    



In [ ]:
import glob
from torch.utils.data import random_split
from typing import Tuple, List

def create_splits(data_dir: str, val_percent: float = 0.2, seed: int = 42) -> Tuple[List[str], List[str]]:
    """
    Trova tutti i campioni e li divide in liste di percorsi per training e validation.
    """
    # 1. Trova tutte le directory dei campioni validi
    all_sample_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d) and not os.path.basename(d).startswith('__')])
    print(f"Trovati {len(all_sample_dirs)} campioni totali validi.")

    # 2. Calcola le dimensioni degli split
    n_val = int(len(all_sample_dirs) * val_percent)
    n_train = len(all_sample_dirs) - n_val
    
    # 3. Esegui lo split
    generator = torch.Generator().manual_seed(seed)
    train_dirs, val_dirs = random_split(all_sample_dirs, [n_train, n_val], generator=generator)
    
    print(f"Split creato: {len(train_dirs)} campioni per training, {len(val_dirs)} per validazione.")
    
    # Restituisce le liste di percorsi
    return list(train_dirs), list(val_dirs)




Trovati 931 campioni totali validi.
Split creato: 745 campioni per training, 186 per validazione.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np

# Importiamo la libreria per i modelli di segmentazione
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss

def main():
    # --- 1. SETUP DEI PARAMETRI E DEI DATALOADER ---
    # (Assumiamo che le celle precedenti con la classe Dataset e lo split siano già state eseguite)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device in uso: {device}")

    # Usiamo i parametri che hanno funzionato meglio nella nostra baseline
    BATCH_SIZE = 8  # Proviamo con 4, dovrebbe reggere con ResNet34 e questa risoluzione
    OUTPUT_SIZE = (256, 512)

    # I valori di mean/std devono essere già stati calcolati
    # Se non lo sono, esegui la cella che li calcola
    # DATASET_MEAN = [0.4278, 0.4464, 0.4231] 
    # DATASET_STD = [0.2543, 0.2535, 0.2567]

    # Creiamo le istanze del dataset e dei dataloader
    # (Assumiamo che train_dirs e val_dirs esistano dalla cella dello split)
    train_dirs, val_dirs = create_splits(DATASET_PATH)
    train_dataset = SegmentationDataset(train_dirs, OUTPUT_SIZE, DATASET_MEAN, DATASET_STD, is_train=True)
    val_dataset = SegmentationDataset(val_dirs, OUTPUT_SIZE, DATASET_MEAN, DATASET_STD, is_train=False)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


    # --- 2. DEFINIZIONE DEL MODELLO, DELLA LOSS E DELL'OTTIMIZZATORE ---

    # Usiamo il modello pre-addestrato dalla libreria SMP
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=3,
        classes=9,
    ).to(device)

    # Definiamo l'ottimizzatore
    optimizer = optim.Adam(model.parameters(), lr=1e-4) # Partiamo con un learning rate più basso per il fine-tuning

    # Definiamo le due componenti della nostra loss
    criterion_ce = nn.CrossEntropyLoss(ignore_index=0)
    criterion_dice = DiceLoss(mode='multiclass') # SMP si aspetta i logits, quindi non serve from_logits=True

    # Definiamo la nostra loss combinata
    def combined_loss(pred, target):
        # Diamo un po' più di peso alla Dice Loss per combattere lo sbilanciamento
        return 0.4 * criterion_ce(pred, target) + 0.6 * criterion_dice(pred, target)


    # --- 3. FUNZIONI DI VALUTAZIONE (invariate) ---

    def compute_all_iou(mask_true, mask_pred, num_labels):
        iou_scores = np.zeros(num_labels)
        for label in range(1, num_labels + 1):
            intersection = np.sum((mask_true == label) & (mask_pred == label))
            union = np.sum((mask_true == label) | (mask_pred == label))
            if union == 0:
                iou_scores[label-1] = np.nan
            else:
                iou_scores[label-1] = intersection / union
        return iou_scores

    def evaluate(model, loader, device, num_classes):
        model.eval()
        all_ious = []
        with torch.no_grad():
            for images, masks in tqdm(loader, desc="Validazione"):
                images = images.to(device)
                masks_np = masks.numpy()
                outputs = model(images)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                for i in range(images.size(0)):
                    iou_scores = compute_all_iou(masks_np[i], preds[i], num_classes - 1)
                    all_ious.append(iou_scores)
        mean_iou = np.nanmean(all_ious)
        return mean_iou


    # --- 4. CICLO DI TRAINING E VALIDAZIONE ---

    num_epochs = 20 # Aumentiamo le epoche, il fine-tuning può essere più lungo
    best_iou = -1.0
    model_save_path = 'best_model_advanced.pth'

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Training]"):
            images = images.to(device)
            masks = masks.to(device, dtype=torch.long)
            
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(images)
            
            # Usa la nostra NUOVA loss combinata
            loss = combined_loss(outputs, masks)
            
            # Backward pass e ottimizzazione
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        epoch_loss = running_loss / len(train_loader)
        
        # Validazione alla fine di ogni epoca
        val_iou = evaluate(model, val_loader, device, num_classes=9)
        
        print(f"Epoch {epoch+1}/{num_epochs} -> Training Loss: {epoch_loss:.4f} | Validation mIoU: {val_iou:.4f}")
        
        # Salva il modello migliore
        if val_iou > best_iou:
            best_iou = val_iou
            torch.save(model.state_dict(), model_save_path)
            print(f"🎉 Nuovo modello migliore salvato in '{model_save_path}' con mIoU: {best_iou:.4f}")
    
if __name__ == "__main__":
    main()

Device in uso: cuda
Trovati 931 campioni totali validi.
Split creato: 745 campioni per training, 186 per validazione.


Epoch 1/20 [Training]:   0%|          | 0/94 [00:10<?, ?it/s]


RuntimeError: DataLoader worker (pid(s) 3784, 23144) exited unexpectedly